# 73 - Florence-2 マルチタスク画像理解

## 概要
Microsoft Florence-2 を使い、複数粒度のキャプション生成に加え、OCR（テキスト抽出）と物体検出を評価する。
BLIP-2（Notebook 72）との比較を行い、マルチタスクモデルの有用性を検証する。

## Florence-2 の特徴
- **DaViT** 画像エンコーダ + seq2seq デコーダ
- **軽量**: ~1.5GB VRAM（Large モデル）
- **マルチタスク**: 単一モデルで多数のビジョンタスクに対応
  - `<CAPTION>`: 簡潔なキャプション
  - `<DETAILED_CAPTION>`: 詳細なキャプション
  - `<MORE_DETAILED_CAPTION>`: より詳細なキャプション
  - `<OCR>`: 画像内テキストの抽出
  - `<OD>`: 物体検出
- **制約**: 英語のみ

## 評価
- Self-Retrieval Score（SigLIP 2 Largeで計算）
- キャプション粒度の比較（CAPTION / DETAILED / MORE_DETAILED）
- OCR結果の確認（カンファレンス画像のスライドテキスト）
- 物体検出結果の確認
- BLIP-2 との速度・品質比較

In [ ]:
import json
import time
from datetime import datetime
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.notebook import tqdm
from transformers import AutoModelForCausalLM, AutoProcessor

## 1. 設定とデータ読み込み

In [ ]:
DB_PATH = Path("../data/images.duckdb")
FLORENCE_MODEL = "microsoft/Florence-2-large-ft"
SIGLIP2_MODEL = "google/siglip2-large-patch16-256"

# 画像カタログの読み込み
conn = duckdb.connect(str(DB_PATH), read_only=True)

catalog_df = conn.execute("""
    SELECT id, file_path, category, file_name
    FROM image_catalog
    ORDER BY id
""").fetchdf()

# SigLIP 2 Large の画像埋め込みを読み込み（評価用）
embeddings_df = conn.execute("""
    SELECT c.id, e.embedding
    FROM image_catalog c
    JOIN image_embeddings e ON c.id = e.id
    WHERE e.model_name = ?
    ORDER BY c.id
""", [SIGLIP2_MODEL]).fetchdf()
conn.close()

image_embeddings = np.array(embeddings_df['embedding'].tolist(), dtype=np.float32)
norms = np.linalg.norm(image_embeddings, axis=1, keepdims=True)
image_embeddings_norm = image_embeddings / np.clip(norms, 1e-8, None)

print(f"Total images: {len(catalog_df)}")
print(f"Image embeddings: {image_embeddings.shape}")

## 2. Florence-2 モデルの読み込み

In [ ]:
print(f"Loading Florence-2 model: {FLORENCE_MODEL}")

processor = AutoProcessor.from_pretrained(FLORENCE_MODEL, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    FLORENCE_MODEL,
    torch_dtype=torch.float16,
    trust_remote_code=True,
).to("cuda")

print(f"Model loaded.")
if torch.cuda.is_available():
    print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")

In [ ]:
def run_florence2(image: Image.Image, task: str, text_input: str = "") -> dict:
    """Florence-2 でタスクを実行する。"""
    prompt = task + text_input
    inputs = processor(text=prompt, images=image, return_tensors="pt").to(
        model.device, dtype=torch.float16
    )
    
    with torch.no_grad():
        generated_ids = model.generate(
            input_ids=inputs["input_ids"],
            pixel_values=inputs["pixel_values"],
            max_new_tokens=1024,
            num_beams=3,
            early_stopping=True,
        )
    
    generated_text = processor.batch_decode(generated_ids, skip_special_tokens=False)[0]
    result = processor.post_process_generation(
        generated_text, task=task, image_size=(image.width, image.height)
    )
    return result

# テスト
test_img = Image.open(catalog_df.iloc[0]['file_path']).convert('RGB')
test_result = run_florence2(test_img, "<CAPTION>")
print(f"Test result: {test_result}")

## 3. マルチ粒度キャプション生成

In [ ]:
# 3つの粒度でキャプション生成
caption_tasks = {
    "caption": "<CAPTION>",
    "detailed": "<DETAILED_CAPTION>",
    "more_detailed": "<MORE_DETAILED_CAPTION>",
}

all_captions = {key: [] for key in caption_tasks}

print(f"Generating captions for {len(catalog_df)} images x {len(caption_tasks)} levels...")
start_time = time.time()

for idx in tqdm(range(len(catalog_df)), desc="Captioning"):
    row = catalog_df.iloc[idx]
    img = Image.open(row['file_path']).convert('RGB')
    
    for key, task in caption_tasks.items():
        result = run_florence2(img, task)
        caption = result.get(task, "")
        all_captions[key].append(caption)

elapsed_caption = time.time() - start_time
print(f"\nTotal time: {elapsed_caption:.1f}s ({elapsed_caption/len(catalog_df):.2f}s per image)")
print(f"Speed: {len(catalog_df) * len(caption_tasks) / elapsed_caption:.1f} captions/sec")

In [ ]:
# サンプルキャプション表示
print("=" * 80)
print("Sample Captions by Granularity")
print("=" * 80)

for cat in catalog_df['category'].unique():
    cat_indices = catalog_df[catalog_df['category'] == cat].index.tolist()[:2]
    print(f"\n--- {cat} ---")
    for idx in cat_indices:
        print(f"  [{catalog_df.iloc[idx]['file_name']}]")
        for key in caption_tasks:
            cap = all_captions[key][idx]
            print(f"    {key:15s}: {cap[:120]}")
        print()

In [ ]:
# キャプションの統計
print("=" * 60)
print("Caption Statistics")
print("=" * 60)

for key in caption_tasks:
    lengths = [len(c.split()) for c in all_captions[key]]
    unique_ratio = len(set(all_captions[key])) / len(all_captions[key])
    print(f"\n{key}:")
    print(f"  Word count — Mean: {np.mean(lengths):.1f}, Std: {np.std(lengths):.1f}, "
          f"Min: {np.min(lengths)}, Max: {np.max(lengths)}")
    print(f"  Unique captions: {len(set(all_captions[key]))} / {len(all_captions[key])} ({unique_ratio*100:.1f}%)")

## 4. OCR（画像内テキスト抽出）

カンファレンス画像のスライドや看板からテキストを抽出する。
BLIP-2 にはない Florence-2 独自の能力。

In [ ]:
# カンファレンスカテゴリの画像でOCRテスト
conference_cats = ["EuroPython2025", "PyConJP2025", "PyConJP2025-PreCampHiroshima"]
conference_indices = catalog_df[catalog_df['category'].isin(conference_cats)].index.tolist()

# サンプル（各カテゴリ5枚）
ocr_sample_indices = []
for cat in conference_cats:
    cat_idx = catalog_df[catalog_df['category'] == cat].index.tolist()[:5]
    ocr_sample_indices.extend(cat_idx)

print(f"Running OCR on {len(ocr_sample_indices)} conference images...")

ocr_results = []
for idx in tqdm(ocr_sample_indices, desc="OCR"):
    row = catalog_df.iloc[idx]
    img = Image.open(row['file_path']).convert('RGB')
    result = run_florence2(img, "<OCR>")
    ocr_text = result.get("<OCR>", "")
    ocr_results.append({
        'index': idx,
        'file_name': row['file_name'],
        'category': row['category'],
        'ocr_text': ocr_text,
    })

print("\nOCR Results:")
for r in ocr_results:
    text_preview = r['ocr_text'][:100] if r['ocr_text'] else "(empty)"
    has_text = "Yes" if r['ocr_text'].strip() else "No"
    print(f"  [{r['category']}] {r['file_name']:40s} | Text: {has_text} | {text_preview}")

In [ ]:
# OCR結果の可視化（テキストが検出された画像）
ocr_with_text = [r for r in ocr_results if r['ocr_text'].strip()]
print(f"Images with detected text: {len(ocr_with_text)} / {len(ocr_results)}")

if ocr_with_text:
    show_count = min(6, len(ocr_with_text))
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    axes = axes.flatten()
    
    for i in range(show_count):
        r = ocr_with_text[i]
        row = catalog_df.iloc[r['index']]
        try:
            img = Image.open(row['file_path'])
            axes[i].imshow(img)
        except Exception:
            pass
        
        text_preview = r['ocr_text'][:80]
        axes[i].set_title(f"[{r['category']}]\nOCR: {text_preview}...", fontsize=8, wrap=True)
        axes[i].axis('off')
    
    for i in range(show_count, len(axes)):
        axes[i].axis('off')
    
    plt.suptitle('Florence-2 OCR Results (Conference Images)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../data/73_florence2_ocr_results.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: data/73_florence2_ocr_results.png")

## 5. 物体検出

In [ ]:
# 各カテゴリ2枚ずつで物体検出
od_sample_indices = []
for cat in catalog_df['category'].unique():
    cat_idx = catalog_df[catalog_df['category'] == cat].index.tolist()[:2]
    od_sample_indices.extend(cat_idx)

print(f"Running object detection on {len(od_sample_indices)} images...")

od_results = []
for idx in tqdm(od_sample_indices, desc="Object Detection"):
    row = catalog_df.iloc[idx]
    img = Image.open(row['file_path']).convert('RGB')
    result = run_florence2(img, "<OD>")
    od_data = result.get("<OD>", {})
    od_results.append({
        'index': idx,
        'file_name': row['file_name'],
        'category': row['category'],
        'od_data': od_data,
    })

In [ ]:
# 物体検出結果の可視化
n_show = min(6, len(od_results))
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for i in range(n_show):
    r = od_results[i]
    row = catalog_df.iloc[r['index']]
    
    try:
        img = Image.open(row['file_path'])
        axes[i].imshow(img)
    except Exception:
        continue
    
    # バウンディングボックスの描画
    od_data = r['od_data']
    if isinstance(od_data, dict) and 'bboxes' in od_data:
        bboxes = od_data['bboxes']
        labels = od_data.get('labels', [''] * len(bboxes))
        
        colors_list = plt.cm.Set2(np.linspace(0, 1, max(len(bboxes), 1)))
        for j, (bbox, label) in enumerate(zip(bboxes, labels)):
            if len(bbox) == 4:
                x1, y1, x2, y2 = bbox
                rect = patches.Rectangle(
                    (x1, y1), x2 - x1, y2 - y1,
                    linewidth=2, edgecolor=colors_list[j % len(colors_list)],
                    facecolor='none'
                )
                axes[i].add_patch(rect)
                axes[i].text(x1, y1 - 5, label, fontsize=7, color='white',
                            bbox=dict(boxstyle='round,pad=0.2', facecolor=colors_list[j % len(colors_list)], alpha=0.8))
        
        n_objects = len(bboxes)
    else:
        n_objects = 0
    
    axes[i].set_title(f"[{r['category']}] {n_objects} objects", fontsize=9)
    axes[i].axis('off')

for i in range(n_show, len(axes)):
    axes[i].axis('off')

plt.suptitle('Florence-2 Object Detection Results', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/73_florence2_object_detection.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: data/73_florence2_object_detection.png")

## 6. Florence-2 のGPUメモリ解放 → SigLIP 2 で評価

In [ ]:
del model, processor
torch.cuda.empty_cache()
print("Florence-2 model released.")

In [ ]:
from image_vector_poc import SigLIP2Embedder

embedder = SigLIP2Embedder(model_name=SIGLIP2_MODEL, device="cuda")
print(f"Loaded: {embedder.model_name} (dim={embedder.embedding_dim})")

## 7. Self-Retrieval Score

In [ ]:
# 各粒度のキャプションをembed
caption_embeddings = {}

for key in caption_tasks:
    print(f"Embedding {key} captions...")
    emb = embedder.embed_texts(all_captions[key])
    caption_embeddings[key] = emb

# Self-Retrieval Score の計算
self_retrieval = {}

print("\n" + "=" * 60)
print("Self-Retrieval Score (Florence-2 Captions)")
print("=" * 60)

for key in caption_tasks:
    scores = np.array([
        np.dot(image_embeddings_norm[i], caption_embeddings[key][i])
        for i in range(len(catalog_df))
    ])
    self_retrieval[key] = scores
    
    print(f"\n{key}:")
    print(f"  Mean:  {scores.mean():.4f}")
    print(f"  Std:   {scores.std():.4f}")
    print(f"  Min:   {scores.min():.4f}")
    print(f"  Max:   {scores.max():.4f}")

In [ ]:
# Retrieval Rank テスト（最良粒度で）
best_key = max(caption_tasks.keys(), key=lambda k: self_retrieval[k].mean())
print(f"Best granularity: {best_key} (mean SR = {self_retrieval[best_key].mean():.4f})")

sim_matrix = np.dot(caption_embeddings[best_key], image_embeddings_norm.T)

ranks = []
for i in range(len(catalog_df)):
    sorted_indices = np.argsort(sim_matrix[i])[::-1]
    rank = np.where(sorted_indices == i)[0][0] + 1
    ranks.append(rank)

ranks = np.array(ranks)

print("\n" + "=" * 60)
print(f"Retrieval Rank ({best_key} Caption → Original Image)")
print("=" * 60)

print(f"  Mean Rank:    {ranks.mean():.1f} / {len(catalog_df)}")
print(f"  Median Rank:  {np.median(ranks):.0f}")
print(f"  Rank@1:  {(ranks == 1).sum()} / {len(catalog_df)} ({(ranks == 1).mean()*100:.1f}%)")
print(f"  Rank<=5:  {(ranks <= 5).sum()} / {len(catalog_df)} ({(ranks <= 5).mean()*100:.1f}%)")
print(f"  Rank<=10: {(ranks <= 10).sum()} / {len(catalog_df)} ({(ranks <= 10).mean()*100:.1f}%)")
print(f"  MRR: {(1.0 / ranks).mean():.4f}")

## 8. BLIP-2 との比較

In [ ]:
# BLIP-2 の結果を読み込み
eval_dir = Path("../data/evaluations")
nb72_files = sorted(eval_dir.glob("72_blip2_captioning_*.json"))

if nb72_files:
    with open(nb72_files[-1]) as f:
        nb72_data = json.load(f)
    print(f"Loaded NB72 results: {nb72_files[-1].name}")
    
    blip2_best_mode = nb72_data['best_mode']
    blip2_sr = nb72_data['self_retrieval'][blip2_best_mode]['mean']
    blip2_mrr = nb72_data['retrieval_rank']['mrr']
    blip2_speed = nb72_data.get('speed_captions_per_sec', 0)
    
    florence_sr = self_retrieval[best_key].mean()
    florence_mrr = (1.0 / ranks).mean()
    florence_speed = len(catalog_df) * len(caption_tasks) / elapsed_caption
    
    print("\n" + "=" * 80)
    print("Comparison: BLIP-2 vs Florence-2")
    print("=" * 80)
    
    print(f"\n{'Metric':<30} {'BLIP-2':>15} {'Florence-2':>15} {'Diff':>12}")
    print("-" * 74)
    print(f"{'Self-Retrieval Score':<30} {blip2_sr:>15.4f} {florence_sr:>15.4f} {florence_sr-blip2_sr:>+12.4f}")
    print(f"{'Retrieval MRR':<30} {blip2_mrr:>15.4f} {florence_mrr:>15.4f} {florence_mrr-blip2_mrr:>+12.4f}")
    print(f"{'Speed (captions/sec)':<30} {blip2_speed:>15.1f} {florence_speed:>15.1f} {florence_speed-blip2_speed:>+12.1f}")
    print(f"{'VRAM (approx)':<30} {'~6 GB':>15} {'~1.5 GB':>15} {'—':>12}")
    print(f"{'Japanese support':<30} {'No':>15} {'No':>15} {'—':>12}")
    print(f"{'OCR capability':<30} {'No':>15} {'Yes':>15} {'—':>12}")
else:
    print("NB72 results not found. Run notebook 72 first for comparison.")

## 9. 可視化

In [ ]:
# 粒度別の Self-Retrieval Score 分布
fig, ax = plt.subplots(figsize=(10, 5))

colors = ['steelblue', 'coral', 'forestgreen']
for i, key in enumerate(caption_tasks):
    ax.hist(self_retrieval[key], bins=30, alpha=0.5, label=key, color=colors[i])
    ax.axvline(self_retrieval[key].mean(), color=colors[i], linestyle='--', linewidth=2)

ax.set_xlabel('Cosine Similarity (Self-Retrieval Score)')
ax.set_ylabel('Count')
ax.set_title('Florence-2 Caption Self-Retrieval Score by Granularity')
ax.legend()

plt.tight_layout()
plt.savefig('../data/73_florence2_self_retrieval_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: data/73_florence2_self_retrieval_distribution.png")

In [ ]:
# 画像とキャプションの視覚的確認
for cat in catalog_df['category'].unique():
    cat_indices = catalog_df[catalog_df['category'] == cat].index.tolist()[:3]
    
    fig, axes = plt.subplots(1, len(cat_indices), figsize=(5 * len(cat_indices), 6))
    if len(cat_indices) == 1:
        axes = [axes]
    
    for i, idx in enumerate(cat_indices):
        row = catalog_df.iloc[idx]
        try:
            img = Image.open(row['file_path'])
            axes[i].imshow(img)
        except Exception:
            axes[i].text(0.5, 0.5, 'Image not found', ha='center', va='center')
        
        caption = all_captions[best_key][idx]
        score = self_retrieval[best_key][idx]
        
        title = f"SR:{score:.3f}\n{caption[:80]}"
        axes[i].set_title(title, fontsize=8, wrap=True)
        axes[i].axis('off')
    
    plt.suptitle(f'Florence-2 ({best_key}) — {cat}', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()
    print()

## 10. DuckDB にキャプションを保存

In [ ]:
conn = duckdb.connect(str(DB_PATH))

conn.execute("""
    CREATE TABLE IF NOT EXISTS image_captions (
        image_id VARCHAR,
        model_name VARCHAR,
        caption_en TEXT,
        caption_ja TEXT,
        prompt_template VARCHAR,
        timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        PRIMARY KEY (image_id, model_name, prompt_template)
    )
""")

conn.execute("DELETE FROM image_captions WHERE model_name = ?", [FLORENCE_MODEL])

for idx in range(len(catalog_df)):
    image_id = catalog_df.iloc[idx]['id']
    caption_en = all_captions[best_key][idx]
    
    conn.execute("""
        INSERT INTO image_captions (image_id, model_name, caption_en, caption_ja, prompt_template)
        VALUES (?, ?, ?, NULL, ?)
    """, [image_id, FLORENCE_MODEL, caption_en, best_key])

conn.commit()

count = conn.execute(
    "SELECT COUNT(*) FROM image_captions WHERE model_name = ?", [FLORENCE_MODEL]
).fetchone()[0]
print(f"Stored {count} captions for model '{FLORENCE_MODEL}'")

# 全モデルの件数
all_counts = conn.execute(
    "SELECT model_name, COUNT(*) FROM image_captions GROUP BY model_name"
).fetchall()
print("\nAll models in image_captions:")
for m, c in all_counts:
    print(f"  {m}: {c}")

conn.close()

## 11. 結果の保存

In [ ]:
output_data = {
    "notebook": "73-florence2-captioning",
    "model_name": FLORENCE_MODEL,
    "eval_model": SIGLIP2_MODEL,
    "timestamp": datetime.now().isoformat(),
    "total_images": len(catalog_df),
    "best_granularity": best_key,
    "self_retrieval": {
        key: {
            "mean": float(self_retrieval[key].mean()),
            "std": float(self_retrieval[key].std()),
        }
        for key in caption_tasks
    },
    "retrieval_rank": {
        "granularity": best_key,
        "mean_rank": float(ranks.mean()),
        "median_rank": float(np.median(ranks)),
        "rank_at_1_ratio": float((ranks == 1).mean()),
        "rank_le_5_ratio": float((ranks <= 5).mean()),
        "rank_le_10_ratio": float((ranks <= 10).mean()),
        "mrr": float((1.0 / ranks).mean()),
    },
    "ocr": {
        "tested_images": len(ocr_sample_indices),
        "images_with_text": len(ocr_with_text),
    },
    "processing_time_seconds": elapsed_caption,
    "speed_captions_per_sec": len(catalog_df) * len(caption_tasks) / elapsed_caption,
}

output_path = Path("../data/evaluations") / f"73_florence2_captioning_{datetime.now().strftime('%Y-%m-%d')}.json"
output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, "w") as f:
    json.dump(output_data, f, indent=2, ensure_ascii=False)

print(f"Results saved to: {output_path}")

## 12. GPU メモリのクリーンアップ

In [ ]:
del embedder

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("GPU memory cleared.")

## まとめ・評価・考察

### 実行状況: SKIP（transformers 5.x 非互換）

Florence-2 (`microsoft/Florence-2-large-ft`) は `trust_remote_code=True` を使用してカスタムコード（`processing_florence2.py`, `configuration_florence2.py`）を読み込むが、これらが **transformers 5.x と非互換** であり、実行できなかった。

**確認されたエラー:**
1. `AttributeError: TokenizersBackend has no attribute additional_special_tokens`
   - `processing_florence2.py` 内で `tokenizer.additional_special_tokens` を直接参照している箇所
   - `getattr()` に修正しても次のエラーが発生
2. `AttributeError: 'Florence2LanguageConfig' object has no attribute 'forced_bos_token_id'`
   - `configuration_florence2.py` のモデル設定クラスが transformers 5.x のインターフェース変更に対応していない
   - より深い互換性問題であり、パッチ適用では対処不可

**対処方針:**
- Florence-2 チームが transformers 5.x 対応版をリリースするまで待機
- または `transformers<5.0.0` の別環境で実行（本プロジェクトは `transformers>=5.0.0` が必要）

### Florence-2 の想定される特徴（参考情報）

| 項目 | 仕様 |
|------|------|
| モデルサイズ | ~1.5GB VRAM（大幅に軽量） |
| 言語 | 英語のみ |
| マルチタスク | キャプション（3粒度）、OCR、物体検出 |
| アーキテクチャ | DaViT encoder + seq2seq decoder |

Florence-2 の最大の特徴はマルチタスク能力（特にOCR）と軽量さであり、カンファレンス画像のスライドテキスト抽出等に活用可能であった。今回の実験シリーズでは BLIP-2（NB72）と Qwen2.5-VL/InternVL2.5（NB74-75）で代替的に評価を進める。